# Data Quality Report — BC_A&A_with_ATD.csv

This notebook runs before any feature engineering or modeling. Its sole purpose is to **audit the raw dataset** and surface issues that could silently corrupt downstream work.

## Checks Performed
| # | Check | What it catches |
|---|-------|-----------------|
| 1 | ATD reconciliation | Discrepancies between preexisting `ATD` column and freshly calculated value |
| 2 | Timezone inference | Whether local timestamps are consistently UTC−6 (Mexico standard time) |
| 3 | Timestamp ordering | Trips where `eater_request → restaurant_offered → order_final_state` is violated |
| 4 | Duplicate IDs | Repeated `delivery_trip_uuid`, `workflow_uuid`, or full-row duplicates |
| 5 | Missing values | Per-column null rates, and whether missingness is random or clustered |
| 6 | Numeric sanity | Negative/zero ATD, impossible distances, extreme outliers |
| 7 | Categorical integrity | Unexpected category values, single-value columns |
| 8 | Summary scorecard | Pass / Warn / Fail per check with recommended actions |

---
## Setup & Load

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (14, 4)})

RAW_PATH = Path('../data/raw/BC_A&A_with_ATD.csv')

# Scorecard: populated throughout the notebook
scorecard = []

In [ ]:
dtype_map = {
    'region': 'category', 'territory': 'category',
    'country_name': 'category', 'courier_flow': 'category',
    'geo_archetype': 'category', 'merchant_surface': 'category',
}
parse_cols = [
    'restaurant_offered_timestamp_utc',
    'order_final_state_timestamp_local',
    'eater_request_timestamp_local',
]

df = pd.read_csv(
    RAW_PATH,
    dtype=dtype_map,
    parse_dates=parse_cols,
    na_values=['\\N', 'NULL', 'None', ''],
)

print(f'Rows    : {len(df):,}')
print(f'Columns : {df.shape[1]}')
print(f'Memory  : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
df.dtypes.to_frame('dtype')

---
## Check 1 — ATD Reconciliation

The `ATD` column should equal `(order_final_state_local + UTC_offset − restaurant_offered_utc)` in minutes.  
We try **UTC−6** (Mexico standard time, year-round since 2022) and also detect the empirical offset in case some rows use a different timezone.

In [ ]:
UTC_OFFSET_H = 6  # hours to add to local timestamps to convert them to UTC

mask_ts = (
    df['restaurant_offered_timestamp_utc'].notna() &
    df['order_final_state_timestamp_local'].notna() &
    df['ATD'].notna()
)

df_ts = df[mask_ts].copy()

df_ts['ATD_calc'] = (
    (df_ts['order_final_state_timestamp_local'] + pd.Timedelta(hours=UTC_OFFSET_H)
     - df_ts['restaurant_offered_timestamp_utc'])
    .dt.total_seconds() / 60
)

df_ts['ATD_residual'] = df_ts['ATD'] - df_ts['ATD_calc']

res = df_ts['ATD_residual']
print('ATD residual = provided − computed (UTC−6)')
print(f'  rows evaluated : {len(df_ts):,}')
print(f'  mean           : {res.mean():.6f} min')
print(f'  std            : {res.std():.6f} min')
print(f'  max abs        : {res.abs().max():.6f} min')
print(f'  |residual|<0.1 : {(res.abs() < 0.1).mean()*100:.3f}% of rows')
print(f'  |residual|>1   : {(res.abs() > 1).sum():,} rows ({(res.abs() > 1).mean()*100:.3f}%)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 4))

# Residual distribution
clipped = res.clip(-60, 60)
axes[0].hist(clipped, bins=100, color='steelblue', edgecolor='none')
axes[0].axvline(0, color='red', linewidth=1.5)
axes[0].set_title('ATD Residual Distribution (clipped ±60 min)')
axes[0].set_xlabel('Provided ATD − Computed ATD (min)')

# Provided vs computed scatter
sample = df_ts.sample(min(20_000, len(df_ts)), random_state=42)
axes[1].scatter(sample['ATD_calc'], sample['ATD'], alpha=0.05, s=5, color='teal')
lim = max(sample['ATD'].quantile(0.99), sample['ATD_calc'].quantile(0.99))
axes[1].plot([0, lim], [0, lim], 'r--', linewidth=1.5)
axes[1].set_xlabel('ATD computed (min)')
axes[1].set_ylabel('ATD provided (min)')
axes[1].set_title('Provided vs Computed ATD')

# Rows with large discrepancy
large_res = df_ts[res.abs() > 60]
if len(large_res) > 0:
    large_res['ATD_residual'].hist(bins=40, ax=axes[2], color='tomato', edgecolor='none')
    axes[2].set_title(f'Large residuals (|err|>60 min) — {len(large_res):,} rows')
    axes[2].set_xlabel('Residual (min)')
else:
    axes[2].text(0.5, 0.5, 'No large residuals\n(|err| > 60 min)', ha='center', va='center',
                 fontsize=13, color='seagreen', transform=axes[2].transAxes)
    axes[2].set_title('Large residuals (|err| > 60 min)')

plt.suptitle('Check 1 — ATD Reconciliation', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Show rows with large discrepancies for inspection
if len(large_res) > 0:
    print(f'Sample of {min(10, len(large_res))} rows with |ATD residual| > 60 min:')
    display(large_res[[
        'delivery_trip_uuid', 'territory', 'courier_flow',
        'restaurant_offered_timestamp_utc', 'order_final_state_timestamp_local',
        'ATD', 'ATD_calc', 'ATD_residual'
    ]].head(10).round(2))

pct_ok = (res.abs() < 0.1).mean() * 100
status = 'PASS' if pct_ok > 99 else ('WARN' if pct_ok > 95 else 'FAIL')
scorecard.append({
    'check': '1. ATD Reconciliation',
    'status': status,
    'detail': f'{pct_ok:.2f}% of rows match within 0.1 min using UTC−6',
})
print(f'\n[{status}] ATD reconciliation: {pct_ok:.2f}% exact match')

---
## Check 2 — Timezone Inference

We empirically derive the UTC offset for each timestamp pair rather than assuming UTC−6.  
This catches:
- Rows from states that historically observed different offsets
- DST artifacts from data collected before 2022
- Encoding errors where local time was stored as UTC (offset = 0)

In [ ]:
# --- Offset: order_final_state_local vs restaurant_offered_utc ---
# If local = UTC-6, then: local + 6h ≈ utc  →  utc - local ≈ 6h
# We compute utc_offered - local_final_state and look at the distribution of hours

df_tz = df[
    df['restaurant_offered_timestamp_utc'].notna() &
    df['order_final_state_timestamp_local'].notna() &
    df['eater_request_timestamp_local'].notna()
].copy()

# Empirical offset = (local + ATD_seconds) - utc_offered  →  infer offset from eater_request
# Simpler: compare restaurant_offered_utc with eater_request_local
# If they represent the same event shifted by UTC offset, diff should be close to ±offset
# Better: look at integer-hour difference between utc and local for same-event timestamps

# For each row: inferred offset = round(utc_offered - eater_request_local) to nearest hour
raw_offset_req = (
    df_tz['restaurant_offered_timestamp_utc']
    - df_tz['eater_request_timestamp_local']
)
# This mixes actual elapsed time + tz offset; instead look at floor-hour difference
# Approach: subtract known ATD to isolate the offset component using final_state
# The cleanest indicator: look at the hour-of-day difference between utc and local columns

utc_hour   = df_tz['restaurant_offered_timestamp_utc'].dt.hour
local_hour = df_tz['eater_request_timestamp_local'].dt.hour

# Raw hour difference (mod 24 to handle midnight wrapping)
# We expect utc_hour - local_hour ≈ +6 for UTC-6
hour_diff = (utc_hour - local_hour).mod(24)
# Values near 18 (= -6 mod 24) indicate UTC-6 correctly;
# let's map back: if diff > 12, subtract 24
hour_diff_signed = hour_diff.apply(lambda x: x - 24 if x > 12 else x)

print('Empirical UTC offset distribution (UTC_hour − local_hour, restaurant_offered vs eater_request):')
print('Expected value: +6 (meaning local is UTC−6)')
print()
vc = hour_diff_signed.value_counts().sort_index()
display(vc.rename('count').to_frame().assign(pct=(vc / len(vc.index.map(lambda _: 1)) * 100).apply(lambda _: None)))
display(hour_diff_signed.value_counts(normalize=True).mul(100).round(2).rename('% of rows').to_frame())

In [ ]:
# Cleaner approach: for rows where ATD is provided, back-calculate the implied offset
# ATD = (local_final + offset_h - utc_offered) / 60
# => offset_h = ATD/60 - (local_final - utc_offered)
# but ATD and the diff are in minutes — let's work in seconds

df_tz2 = df_tz[df_tz['ATD'].notna() & (df_tz['ATD'] > 0) & (df_tz['ATD'] < 240)].copy()

# (local_final - utc_offered) in hours (naive subtraction, ignoring real timezone)
naive_diff_h = (
    df_tz2['order_final_state_timestamp_local']
    - df_tz2['restaurant_offered_timestamp_utc']
).dt.total_seconds() / 3600

# Implied offset = ATD_hours - naive_diff_h
# Because: ATD = (local + offset - utc) → offset = ATD - (local - utc)
implied_offset_h = (df_tz2['ATD'] / 60) - naive_diff_h

# Round to nearest 0.5h for distribution
implied_offset_rounded = (implied_offset_h * 2).round() / 2

print('Implied UTC offset derived from ATD + timestamps (expected: 6.0):')
display(
    implied_offset_rounded.value_counts(normalize=True)
    .mul(100).round(3)
    .rename('% of rows')
    .sort_index()
    .to_frame()
)

dominant_offset = implied_offset_rounded.mode()[0]
anomalous_tz = (implied_offset_rounded != dominant_offset).sum()
print(f'\nDominant offset : {dominant_offset:+.1f} h')
print(f'Anomalous rows  : {anomalous_tz:,} ({anomalous_tz/len(df_tz2)*100:.2f}%)')

In [ ]:
# Territory-level timezone breakdown
# Use the implied_offset computed above (rounded to nearest integer hour)
_implied_int = _implied.round().astype(int)

territory_tz_dist = (
    pd.DataFrame({
        'territory': df_tz2.loc[_implied_int.index, 'territory'].values,
        'offset_h' : _implied_int.values,
    })
    .groupby(['territory', 'offset_h'])
    .size()
    .rename('rows')
    .reset_index()
)

# Dominant offset and row share per territory
territory_tz_summary = (
    territory_tz_dist
    .sort_values('rows', ascending=False)
    .groupby('territory', sort=False)
    .apply(lambda g: pd.Series({
        'dominant_offset_h' : g.loc[g['rows'].idxmax(), 'offset_h'],
        'dominant_pct'      : g['rows'].max() / g['rows'].sum() * 100,
        'n_offsets_observed': g['offset_h'].nunique(),
        'total_rows'        : g['rows'].sum(),
    }))
    .reset_index()
    .sort_values('dominant_offset_h')
)

print('Territory → dominant UTC offset mapping:')
print('(dominant_offset_h = hours to add to local time to obtain UTC)')
display(territory_tz_summary.round(1))

# Show rows whose implied offset differs from the territory dominant
mismatch_by_territory = (
    pd.DataFrame({
        'territory': df_tz2.loc[_implied_int.index, 'territory'].values,
        'implied'  : _implied_int.values,
    })
    .merge(
        territory_tz_summary[['territory', 'dominant_offset_h']],
        on='territory', how='left'
    )
    .assign(is_mismatch=lambda x: x['implied'] != x['dominant_offset_h'])
    .groupby('territory')['is_mismatch']
    .agg(['sum', 'mean'])
    .rename(columns={'sum': 'mismatch_rows', 'mean': 'mismatch_pct'})
    .assign(mismatch_pct=lambda x: (x['mismatch_pct'] * 100).round(2))
    .sort_values('mismatch_rows', ascending=False)
)
print('\nIntra-territory timezone inconsistency (rows whose offset ≠ territory dominant):')
display(mismatch_by_territory)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Distribution of implied offsets
implied_offset_rounded.value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Implied UTC Offset Distribution')
axes[0].set_xlabel('Hours (+ means local is behind UTC)')
axes[0].set_ylabel('Row Count')
axes[0].tick_params(axis='x', rotation=0)

# Anomalous-offset rows by territory
anomalous_mask = implied_offset_rounded != dominant_offset
if anomalous_mask.sum() > 0:
    anom_by_territory = (
        df_tz2[anomalous_mask.values]
        .groupby('territory', observed=True)
        .size()
        .sort_values(ascending=False)
    )
    anom_by_territory.plot(kind='bar', ax=axes[1], color='tomato', edgecolor='none')
    axes[1].set_title('Anomalous Timezone Rows by Territory')
    axes[1].tick_params(axis='x', rotation=35)
else:
    axes[1].text(0.5, 0.5, 'All rows consistent\nwith dominant offset',
                 ha='center', va='center', fontsize=13, color='seagreen',
                 transform=axes[1].transAxes)
    axes[1].set_title('Anomalous Timezone Rows by Territory')

plt.suptitle('Check 2 — Timezone Inference', fontweight='bold')
plt.tight_layout()
plt.show()

status = 'PASS' if dominant_offset == 6.0 and anomalous_tz / len(df_tz2) < 0.01 else \
         'WARN' if dominant_offset == 6.0 else 'FAIL'
scorecard.append({
    'check': '2. Timezone Consistency',
    'status': status,
    'detail': f'Dominant offset = {dominant_offset:+.1f} h. Anomalous rows: {anomalous_tz:,} ({anomalous_tz/len(df_tz2)*100:.2f}%)',
})
print(f'[{status}] Dominant offset={dominant_offset:+.1f}h, anomalous={anomalous_tz:,} rows')

---
## Check 3 — Timestamp Ordering

The logical event sequence must be:
```
eater_request_local  <  restaurant_offered_utc (adjusted)  <  order_final_state_local
```
Violations indicate clock skew, data pipeline bugs, or test records.

In [ ]:
df_ord = df[
    df['restaurant_offered_timestamp_utc'].notna() &
    df['order_final_state_timestamp_local'].notna() &
    df['eater_request_timestamp_local'].notna()
].copy()

offset = pd.Timedelta(hours=6)

# Convert all to a common reference (UTC) for comparison
offered_utc    = df_ord['restaurant_offered_timestamp_utc']
request_utc    = df_ord['eater_request_timestamp_local']  + offset
final_utc      = df_ord['order_final_state_timestamp_local'] + offset

violations = {
    'request ≥ offered (offered before request)' : (request_utc >= offered_utc).sum(),
    'offered ≥ final   (delivered before offered)': (offered_utc >= final_utc).sum(),
    'request ≥ final   (delivered before request)' : (request_utc >= final_utc).sum(),
    'final − offered < 0 (negative ATD)'           : (final_utc < offered_utc).sum(),
    'final − offered > 24h (ATD > 1 day)'          : ((final_utc - offered_utc).dt.total_seconds() > 86400).sum(),
}

viol_df = pd.DataFrame(list(violations.items()), columns=['Violation', 'Row Count'])
viol_df['% of rows'] = (viol_df['Row Count'] / len(df_ord) * 100).round(4)
print(f'Timestamp ordering violations (out of {len(df_ord):,} rows with all 3 timestamps):')
display(viol_df)

In [ ]:
# Show examples of the most critical violation (offered ≥ final)
neg_atd_mask = offered_utc >= final_utc
if neg_atd_mask.sum() > 0:
    print(f'Sample rows where offered ≥ final (negative/zero ATD):')
    display(df_ord[neg_atd_mask.values][[
        'delivery_trip_uuid', 'territory', 'courier_flow',
        'restaurant_offered_timestamp_utc',
        'order_final_state_timestamp_local', 'ATD',
    ]].head(10))

total_violations = sum(v for v in violations.values())
status = 'PASS' if total_violations == 0 else \
         'WARN' if total_violations / len(df_ord) < 0.01 else 'FAIL'
scorecard.append({
    'check': '3. Timestamp Ordering',
    'status': status,
    'detail': f'Total violations: {total_violations:,} across all ordering checks',
})
print(f'[{status}] Total timestamp ordering violations: {total_violations:,}')

---
## Check 4 — Duplicate ID Detection

- `delivery_trip_uuid` — should be unique (one row per trip)
- `workflow_uuid` — may legitimately repeat (re-dispatch of same order)
- Full-row duplicates — indicates ETL double-write

In [ ]:
id_cols = ['delivery_trip_uuid', 'workflow_uuid', 'driver_uuid']

for col in id_cols:
    total     = df[col].notna().sum()
    n_unique  = df[col].nunique()
    n_dup     = total - n_unique
    pct_dup   = n_dup / total * 100 if total > 0 else 0
    top_count = df[col].value_counts().iloc[0] if n_unique > 0 else 0
    print(f'{col}:')
    print(f'  total non-null : {total:>10,}')
    print(f'  unique         : {n_unique:>10,}')
    print(f'  duplicates     : {n_dup:>10,}  ({pct_dup:.3f}%)')
    print(f'  max occurrences: {top_count:>10,}')
    print()

In [ ]:
# delivery_trip_uuid must be unique — show duplicates if they exist
dup_trip = df[df['delivery_trip_uuid'].notna() &
              df.duplicated('delivery_trip_uuid', keep=False)]

if len(dup_trip) > 0:
    print(f'delivery_trip_uuid duplicates: {len(dup_trip):,} rows ({len(dup_trip)/len(df)*100:.3f}%)')
    print('Sample duplicated trips:')
    display(dup_trip.sort_values('delivery_trip_uuid').head(10))
else:
    print('✓ delivery_trip_uuid is fully unique — no duplicate trips.')

# workflow_uuid duplicates (expected — re-dispatches)
dup_workflow = df['workflow_uuid'].value_counts()
multi_trip_orders = (dup_workflow > 1).sum()
print(f'\nworkflow_uuid appearing >1 time (re-dispatched orders): {multi_trip_orders:,}')
print('Distribution of trips per workflow:')
display(dup_workflow.value_counts().rename_axis('trips_per_workflow').rename('n_workflows').head(10).to_frame())

In [ ]:
# Full-row duplicates
full_dups = df.duplicated().sum()
print(f'Full-row duplicates: {full_dups:,} ({full_dups/len(df)*100:.4f}%)')
if full_dups > 0:
    print('Sample duplicated rows:')
    display(df[df.duplicated(keep=False)].head(6))

n_dup_trips = dup_trip['delivery_trip_uuid'].nunique() if len(dup_trip) > 0 else 0
status = 'PASS' if n_dup_trips == 0 and full_dups == 0 else \
         'WARN' if (n_dup_trips + full_dups) / len(df) < 0.01 else 'FAIL'
scorecard.append({
    'check': '4. Duplicate IDs',
    'status': status,
    'detail': f'Dup delivery_trip_uuid: {n_dup_trips:,} | Full-row dups: {full_dups:,} | Multi-trip workflows: {multi_trip_orders:,}',
})
print(f'[{status}] Duplicate check complete')

---
## Check 5 — Missing Values

In [ ]:
missing = (
    df.isnull().sum()
    .rename('n_missing')
    .to_frame()
    .assign(pct=lambda x: (x['n_missing'] / len(df) * 100).round(4))
    .sort_values('n_missing', ascending=False)
)
print('Missing value profile:')
display(missing)

fig, ax = plt.subplots(figsize=(12, 4))
missing['pct'].plot(kind='bar', ax=ax, color='coral', edgecolor='none')
ax.set_title('Missing Value Rate by Column (%)', fontweight='bold')
ax.set_ylabel('% Missing')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
# Is distance missingness correlated? (both pickup and dropoff missing together?)
pickup_na  = df['pickup_distance'].isna()
dropoff_na = df['dropoff_distance'].isna()

both_na   = (pickup_na & dropoff_na).sum()
only_pick = (pickup_na & ~dropoff_na).sum()
only_drop = (~pickup_na & dropoff_na).sum()

print('Distance missing pattern:')
print(f'  Both missing            : {both_na:,} ({both_na/len(df)*100:.3f}%)')
print(f'  Only pickup missing     : {only_pick:,}')
print(f'  Only dropoff missing    : {only_drop:,}')

# Is missingness random or clustered by segment?
miss_by_segment = (
    df.groupby(['territory', 'courier_flow'], observed=True)['pickup_distance']
    .apply(lambda x: x.isna().mean() * 100)
    .rename('missing_pct')
    .sort_values(ascending=False)
    .head(15)
)
print('\nTop 15 segments by pickup_distance missing rate:')
display(miss_by_segment.round(2).to_frame())

In [ ]:
# ATD missing
atd_missing = df['ATD'].isna().sum()
dist_missing = pickup_na.sum()
driver_missing = df['driver_uuid'].isna().sum()

worst_pct = missing['pct'].max()
status = 'PASS' if worst_pct < 5 else ('WARN' if worst_pct < 20 else 'FAIL')
scorecard.append({
    'check': '5. Missing Values',
    'status': status,
    'detail': f'ATD: {atd_missing:,} | pickup_distance: {dist_missing:,} | driver_uuid: {driver_missing:,} | worst col: {worst_pct:.2f}%',
})
print(f'[{status}] Worst missing rate: {worst_pct:.2f}%')

---
## Check 6 — Numeric Sanity (ATD & Distances)

In [ ]:
atd = df['ATD'].dropna()

atd_checks = {
    'ATD is null'          : df['ATD'].isna().sum(),
    'ATD ≤ 0 (impossible)' : (atd <= 0).sum(),
    'ATD < 1 min'          : (atd < 1).sum(),
    'ATD > 120 min'        : (atd > 120).sum(),
    'ATD > 240 min'        : (atd > 240).sum(),
    'ATD > 480 min'        : (atd > 480).sum(),
    'ATD > 1440 min (>1d)' : (atd > 1440).sum(),
}

atd_df = pd.DataFrame(list(atd_checks.items()), columns=['Condition', 'Count'])
atd_df['% of total'] = (atd_df['Count'] / len(df) * 100).round(4)
print('ATD sanity checks:')
display(atd_df)

print(f'\nATD full distribution:')
print(atd.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2))

In [ ]:
for col in ['pickup_distance', 'dropoff_distance']:
    s = df[col].dropna()
    print(f'{col}:')
    print(f'  negative  : {(s < 0).sum():,}')
    print(f'  zero      : {(s == 0).sum():,}')
    print(f'  > 50 km   : {(s > 50).sum():,}  ({(s > 50).mean()*100:.3f}%)')
    print(f'  > 100 km  : {(s > 100).sum():,}  ({(s > 100).mean()*100:.3f}%)')
    print(f'  p99 = {s.quantile(0.99):.2f} km  |  max = {s.max():.2f} km')
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 4))

# ATD full range (log y)
axes[0].hist(atd.clip(0, 500), bins=100, color='steelblue', edgecolor='none', log=True)
axes[0].axvline(0, color='red', linewidth=1.5, label='0 min')
axes[0].set_title('ATD Distribution (0–500 min, log y)')
axes[0].set_xlabel('ATD (min)')
axes[0].legend()

# Pickup distance
p = df['pickup_distance'].dropna()
axes[1].hist(p[p <= p.quantile(0.99)], bins=70, color='teal', edgecolor='none')
axes[1].set_title('Pickup Distance (≤ p99)')
axes[1].set_xlabel('km')

# Dropoff distance
d = df['dropoff_distance'].dropna()
axes[2].hist(d[d <= d.quantile(0.99)], bins=70, color='coral', edgecolor='none')
axes[2].set_title('Dropoff Distance (≤ p99)')
axes[2].set_xlabel('km')

plt.suptitle('Check 6 — Numeric Sanity', fontweight='bold')
plt.tight_layout()
plt.show()

impossible = int((atd <= 0).sum())
status = 'PASS' if impossible == 0 else ('WARN' if impossible / len(df) < 0.001 else 'FAIL')
scorecard.append({
    'check': '6. Numeric Sanity',
    'status': status,
    'detail': f'ATD ≤ 0: {impossible:,} | ATD > 240 min: {int((atd > 240).sum()):,} | negative distances: {int((df["pickup_distance"] < 0).sum()):,}',
})
print(f'[{status}] Numeric sanity check complete')

---
## Check 7 — Categorical Integrity

In [ ]:
cat_cols = ['region', 'territory', 'country_name', 'courier_flow', 'geo_archetype', 'merchant_surface']

cat_report = []
for col in cat_cols:
    vc = df[col].value_counts(dropna=False)
    cat_report.append({
        'column'          : col,
        'n_unique'        : df[col].nunique(),
        'null_count'      : df[col].isna().sum(),
        'top_value'       : vc.index[0],
        'top_pct'         : f"{vc.iloc[0]/len(df)*100:.1f}%",
        'bottom_value'    : vc.index[-1] if len(vc) > 1 else vc.index[0],
        'bottom_count'    : int(vc.iloc[-1]),
        'single_value_col': vc.shape[0] == 1,
    })

cat_df = pd.DataFrame(cat_report)
display(cat_df)

In [ ]:
for col in cat_cols:
    print(f'{col} value counts:')
    vc = df[col].value_counts(dropna=False)
    display(
        vc.to_frame('count')
        .assign(pct=(vc / len(df) * 100).round(2))
    )
    print()

In [ ]:
single_val_cols = [r['column'] for r in cat_report if r['single_value_col']]
null_cat_cols   = [r['column'] for r in cat_report if r['null_count'] > 0]

status = 'FAIL' if single_val_cols else ('WARN' if null_cat_cols else 'PASS')
scorecard.append({
    'check': '7. Categorical Integrity',
    'status': status,
    'detail': f'Single-value cols: {single_val_cols or "none"} | Cols with nulls: {null_cat_cols or "none"}',
})
print(f'[{status}] Categorical integrity check complete')

---
## Check 8 — Summary Scorecard

In [ ]:
score_df = pd.DataFrame(scorecard)

STATUS_ICON = {'PASS': '✅', 'WARN': '⚠️ ', 'FAIL': '❌'}
STATUS_COLOR = {'PASS': 'seagreen', 'WARN': 'goldenrod', 'FAIL': 'tomato'}

print('╔' + '═' * 80 + '╗')
print('║  DATA QUALITY SCORECARD' + ' ' * 57 + '║')
print('╠' + '═' * 80 + '╣')
for _, row in score_df.iterrows():
    icon   = STATUS_ICON.get(row['status'], '?')
    check  = row['check'][:35].ljust(35)
    status = row['status'].ljust(4)
    detail = row['detail'][:38]
    print(f'║  {icon} {check} [{status}]  {detail}  ║')
print('╠' + '═' * 80 + '╣')

n_pass = (score_df['status'] == 'PASS').sum()
n_warn = (score_df['status'] == 'WARN').sum()
n_fail = (score_df['status'] == 'FAIL').sum()
overall = 'PASS' if n_fail == 0 and n_warn == 0 else \
          'WARN' if n_fail == 0 else 'FAIL'
print(f'║  Overall: {STATUS_ICON[overall]} {overall}  |  ✅ {n_pass} passed  ⚠️  {n_warn} warnings  ❌ {n_fail} failures' + ' ' * 20 + '║')
print('╚' + '═' * 80 + '╝')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

colors = [STATUS_COLOR.get(s, 'gray') for s in score_df['status']]
bars   = ax.barh(score_df['check'], [1] * len(score_df), color=colors, edgecolor='none')

for bar, (_, row) in zip(bars, score_df.iterrows()):
    icon = STATUS_ICON.get(row['status'], '?')
    ax.text(
        0.02, bar.get_y() + bar.get_height() / 2,
        f"{icon} {row['status']}  —  {row['detail'][:65]}",
        va='center', ha='left', fontsize=9, color='white', fontweight='bold',
    )

ax.set_xlim(0, 1)
ax.set_xticks([])
ax.set_title('Data Quality Scorecard', fontweight='bold', fontsize=13)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

display(score_df)